In [3]:
import os
from dotenv import load_dotenv
# load all environment variables
load_dotenv()
# Load groq API key into environment variable
os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY_SIMPLE_CHAT_BOT"] = os.getenv("LANGCHAIN_API_KEY_SIMPLE_CHAT_BOT")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "SimpleChatBot"
groq_api_key = os.getenv("GROK_API_KEY")

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F0A43A3E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F0A40F0890>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [29]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
## Create in-memory store for chat histories
store = {}
## Function to get or create chat history for a session
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]
## Wrap LLM with message history functionality
with_message_history = RunnableWithMessageHistory(llm, get_session_history)

In [4]:
config = {
    "configurable" : {"session_id" : "chat1"}
}

In [ ]:
from langchain_core.messages import HumanMessage

ai_response = with_message_history.invoke(HumanMessage(content="Hello, how are you? my name is Mohamed"), config=config)
ai_response.content


"Nice to meet you, Mohamed. I'm a computer program, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or topics you'd like to discuss. How are you doing today, Mohamed?"

In [8]:
ai_response = with_message_history.invoke(HumanMessage(content="Hello, what is my name"), config=config)
ai_response.content

'Your name is Mohamed, we just established that.'

In [ ]:
## change session id lead to new chat history so the bot forgets previous messages
config1 = {
    "configurable" : {"session_id" : "chat2"}
}
ai_response = with_message_history.invoke(HumanMessage(content="Hello, what is my name"), config=config1)
ai_response.content

"I'm happy to chat with you, but unfortunately, I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or retain a memory of previous conversations. Each time you interact with me, it's a new conversation and I don't know who you are or any details about you.\n\nIf you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation."

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

## Define a prompt template with message history placeholder as human message given in form of key and value pair (key is messages)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer the questions to the best of your ability."),
        (MessagesPlaceholder(variable_name="messages"))
    ])

chain = prompt | llm 

In [13]:
chain.invoke({"messages": [HumanMessage(content="Hello, how are you? my name is Mohamed")]})

AIMessage(content="Hello Mohamed, nice to meet you. I'm just a computer program, so I don't have emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 61, 'total_tokens': 113, 'completion_time': 0.061992407, 'completion_tokens_details': None, 'prompt_time': 0.004415435, 'prompt_tokens_details': None, 'queue_time': 0.089285813, 'total_time': 0.066407842}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c1ebf-1b41-78f0-be57-cf904a304893-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 52, 'total_tokens': 113})

In [ ]:
with_message_history = chain, get_session_history)

In [16]:
config = {
    "configurable" : {"session_id" : "chat3"}
}

In [17]:
ai_response = with_message_history.invoke(HumanMessage(content="Hello, how are you? my name is Mohamed"), config=config)
ai_response.content

"Hello Mohamed, I'm doing well, thank you for asking. It's nice to meet you. I'm a helpful assistant, here to provide information and assist with any questions or tasks you may have. How can I help you today?"

In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer the questions to the best of your ability in {language}."),
        (MessagesPlaceholder(variable_name="messages"))
    ])

chain = prompt | llm 

In [19]:
response =chain.invoke({"messages": [HumanMessage(content="Hello, how are you? my name is Mohamed")], "language": "Arabic"})
response.content

'مرحبا محمد، أنني بخير. كيف حالك؟'

In [20]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key = "messages")

In [21]:
config = {
    "configurable" : {"session_id" : "chat4"}
}
ai_response = with_message_history.invoke({'messages' : [HumanMessage(content="Hello, how are you? my name is Mohamed")],
                                           'language' : "Arabic"}, config=config)
ai_response.content

'مرحباً، أنا بخير شكراً. اسمي مساعد، سعيد أن ألتقي بك، انت محمد. كيف حالك؟'

In [24]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage,trim_messages
trimmer = trim_messages(
                        max_tokens = 45,
                        strategy = "last",
                        token_counter = llm,
                        include_system = True,
                        allow_partial = False,
                        start_on = "human"
                        )

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hello, my name is Mohamed"),
    AIMessage(content="hi"),
    HumanMessage(content="Can you help me with something?"),
    AIMessage(content="Sure, what do you need?"),
    HumanMessage(content="what 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!")
]

trimmer.invoke(messages)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Can you help me with something?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure, what do you need?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
## Define a prompt template with message history placeholder as human message given in form of key and value pair (key is messages)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer the questions to the best of your ability."),
        (MessagesPlaceholder(variable_name= "messages"))
    ])

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer)
    | prompt
    | llm
)

response = chain.invoke(
    {
        "messages" : messages + [HumanMessage(content="what is your job?") ],
        "language" : "English"
    }
    )

print(response.content)

My job is to assist and provide information to users by answering their questions to the best of my ability.


In [30]:
from langchain_core.runnables.history import RunnableWithMessageHistory
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key = "messages")
config = {
    "configurable" : {"session_id" : "chat5"}
}
ai_response = with_message_history.invoke({'messages' : [HumanMessage(content="what is my name?")],
                                           'language' : "English"}, config=config)
ai_response.content

"I don't have any information about your name as our conversation has just started. I'm a helpful assistant, and I can assist you with various topics or answer your questions, but I don't have any personal data about you, including your name. If you'd like to share your name with me, I can use it in our conversation to make it more personalized."